In [ ]:
import torch, torch.nn as nn
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

# =============================================================================
# Elementary Cellular Automata (ECA) utilities
# -----------------------------------------------------------------------------
# Conventions:
# - States are 1D binary rows with periodic (circular) boundary conditions.
# - Torch tensors use channel-first format for Conv1d: [B, C, N].
# - We keep CA states as uint8 during simulation for compactness / clarity,
#   then cast to float for the neural-loss targets.
# =============================================================================


def rule_table(rule):
    """
    Build the 8-entry truth table for an ECA rule as a uint8 tensor.

    For ECAs (radius r=1), each cell's next state depends on the triple (L,C,R),
    which encodes to an index in [0..7] as (L<<2) | (C<<1) | R.
    'rule' packs the 8 outputs into its bits.

    Returns
    -------
    tbl : torch.Tensor, shape [8], dtype=uint8
        tbl[idx] = next-state bit for neighborhood encoded by idx.
    """
    return torch.tensor([(rule >> i) & 1 for i in range(8)], dtype=torch.uint8)

def evolve_once(state, tbl):  # state: [B,N] in {0,1}
    """
    Evolve one CA step with periodic boundaries.

    Parameters
    ----------
    state : torch.Tensor, shape [B, N], dtype in {uint8, long}
        Current binary row(s) (0/1). Batch supported.
    tbl   : torch.Tensor, shape [8], dtype=uint8
        Truth table from rule_table(rule).

    Returns
    -------
    next_state : torch.Tensor, shape [B, N], dtype=uint8
    """
    L = torch.roll(state,  1, dims=-1)
    R = torch.roll(state, -1, dims=-1)
    idx = ((L << 2) | (state << 1) | R).long()  # <-- make it Long
    return tbl[idx]  # shape [B,N], dtype stays as tbl's dtype

def jump_ahead(state, tbl, H):
    """
    Compose the local rule H times (explicit rollout).

    Parameters
    ----------
    state : [B, N] uint8
    tbl   : [8] uint8
    H     : int, number of steps to evolve

    Returns
    -------
    state_H : [B, N] uint8
    """
    for _ in range(H):
        state = evolve_once(state, tbl)
    return state

def simulate(rule: int, width=256, steps=256, p_init=0.5, seed=0, init="random") -> torch.Tensor:
    """
    Simulate an ECA space-time diagram.

    Parameters
    ----------
    rule   : int in [0..255]
    width  : number of cells (N)
    steps  : number of time steps (rows)
    p_init : Bernoulli parameter for random init
    seed   : RNG seed for reproducibility
    init   : "random" | "single"
        - "random": Bernoulli(p_init)
        - "single": single 1 at center

    Returns
    -------
    grid : torch.Tensor, shape [steps, width], dtype=uint8
        Row 0 is t=0; periodic boundaries.
    """
    g = torch.Generator().manual_seed(seed)
    if init == "random":
        state = (torch.rand(width, generator=g) < p_init).to(torch.uint8).unsqueeze(0)  # [1,N]
    elif init == "single":
        state = torch.zeros(width, dtype=torch.uint8).unsqueeze(0)
        state[0, width // 2] = 1
    else:
        raise ValueError("init must be 'random' or 'single'")

    tbl = rule_table(rule)
    grid = torch.zeros((steps, width), dtype=torch.uint8)
    grid[0] = state[0]
    for t in range(1, steps):
        state = evolve_once(state, tbl)
        grid[t] = state[0]
    return grid

# =============================================================================
# Dataset builder: (s^t  ->  s^{t+H}) pairs
# -----------------------------------------------------------------------------
# We generate input rows x and labels y on-the-fly by rolling out the true CA.
# x, y are returned as float tensors and channelized for Conv1d: [B, 1, N].
# =============================================================================

def make_batch(B: int, N: int, rule: int, H: int, device: str = "cpu"):
    """
    Create a mini-batch of jump-ahead training pairs.

    Parameters
    ----------
    B, N  : batch size and width
    rule  : ECA rule number
    H     : jump horizon (predict t+H from t)
    device: torch device string

    Returns
    -------
    x : torch.FloatTensor, shape [B, 1, N]
        Input rows at time t (0/1 floats for BCE).
    y : torch.FloatTensor, shape [B, 1, N]
        Target rows at time t+H (0/1 floats).
    """
    # Random 0/1 input rows (long for bit-shifts; we cast later)
    x_bits = torch.randint(0, 2, (B, N), dtype=torch.long, device=device)
    # Ground-truth rollout for the label
    y_bits = jump_ahead(x_bits.clone(), rule_table(rule).to(device), H)

    # Channelize + cast to float for BCEWithLogitsLoss
    x = x_bits.unsqueeze(1).float()  # [B, 1, N]
    y = y_bits.unsqueeze(1).float()  # [B, 1, N]
    return x, y

# =============================================================================
# Shallow CNN for jump-ahead prediction
# -----------------------------------------------------------------------------
# Architecture:
#   Conv1d(1 -> hidden, kernel=2H+1, circular padding, bias=False)
#   ReLU (or Identity for strictly linear mapping on additive rules)
#   Conv1d(hidden -> 1, kernel=1, bias=True)
#
# Intuition:
# - The first (wide) conv gives each output position access to the entire
#   radius-H neighborhood (receptive field = 2H+1), extracting 'hidden'
#   parallel features per position.
# - The 1x1 conv is a per-position linear classifier/regressor that mixes
#   the hidden feature vector down to a single logit (binary state).
# - Using circular padding matches the CA's torus boundary condition.
# - bias=False in the wide conv nudges it toward pure pattern kernels;
#   bias=True in the head lets logits shift as needed.
# =============================================================================

class ShallowCNN(nn.Module):
    def __init__(self, H, hidden=16):
        """
        Parameters
        ----------
        H      : jump horizon; sets receptive field size (2H+1)
        hidden : number of feature channels after the first conv
        """
        super().__init__()
        k = 2*H + 1  # kernel size = receptive field covering radius-H

        # 1) Wide, circular Conv1d:
        #    in_channels=1, out_channels=hidden, kernel=2H+1
        #    padding=k//2 keeps length N; 'circular' enforces periodic BCs.
        self.conv1 = nn.Conv1d(1, hidden, kernel_size=k, padding=k//2,
                               padding_mode='circular', bias=False)

        # 2) Nonlinearity:
        #    ReLU adds expressivity (use nn.Identity() for strictly linear mapping).
        self.act = nn.ReLU()

        # 3) 1x1 output head:
        #    mixes the 'hidden' features at each position into a single logit.
        self.out = nn.Conv1d(
            in_channels=hidden,
            out_channels=1,
            kernel_size=1,
            bias=True
        )
    def forward(self, x):
        """
        Forward pass.

        Parameters
        ----------
        x : torch.FloatTensor, shape [B, 1, N]
            Input rows (0/1 floats).

        Returns
        -------
        logits : torch.FloatTensor, shape [B, 1, N]
            Pre-sigmoid logits for BCEWithLogitsLoss.
        """
        # Shapes:
        # x        : [B, 1, N]
        # h = conv1: [B, hidden, N]  (each channel uses a different radius-H filter)
        # a = act  : [B, hidden, N]
        # y = out  : [B, 1, N]       (per-position linear mix -> single logit)
        h = self.conv1(x)
        a = self.act(h)
        logits = self.out(a)
        return logits


# =============================================================================
# Minimal training loop
# -----------------------------------------------------------------------------
# - Uses BCEWithLogitsLoss (numerically stable sigmoid + BCE).
# - Reports bit accuracy by thresholding sigmoid(logits) at 0.5.
# - Intentional simplicity to highlight reducible vs. irreducible behavior.
# =============================================================================

def train_rule(rule, H=16, N=256, steps=500, B=64, device="cuda"):
    """
    Train ShallowCNN to predict s^{t+H} from s^t for a given ECA rule.

    Parameters
    ----------
    rule  : ECA rule number (e.g., 150, 90, 30)
    H     : jump horizon (controls receptive field = 2H+1)
    N     : width (number of cells)
    steps : optimizer steps
    B     : batch size
    device: 'cuda' or 'cpu'

    Notes
    -----
    - For additive/reducible rules (e.g., 90/150), this shallow model can learn
      the closed-form H-ahead mapping well, even for large H.
    - For irreducible rules (e.g., 30), accuracy will degrade as H grows because
      the true mapping is a composition of the local rule H times (requires depth).
    """
    model = ShallowCNN(H).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.BCEWithLogitsLoss()

    for step in range(steps):
        x, y = make_batch(B, N, rule, H, device)
        logits = model(x)
        loss = loss_fn(logits, y)
        opt.zero_grad(); loss.backward(); opt.step()
        if (step+1) % 25 == 0:
            with torch.no_grad():
                acc = ((logits.sigmoid() > 0.5) == (y > 0.5)).float().mean().item()
            print(f"rule {rule:3d} | H={H:2d} | step {step+1:3d} | loss {loss:.3f} | acc {acc:.3f}")

# =============================================================================
# Optional tips / toggles (uncomment as needed)
# -----------------------------------------------------------------------------
# 1) For purely additive rules (90/150), try a strictly linear model:
#       self.act = nn.Identity()
#    This makes the whole network linear (wide conv + 1x1), aligning with the
#    XOR-parity closed form in GF(2), while still training with real-valued ops.
#
# 2) Inspect learned wide kernel for interpretability:
#       with torch.no_grad():
#           W = model.conv1.weight.squeeze(1)   # [hidden, 2H+1]
#    You can visualize W to see which offsets are emphasized.
#
# 3) Deep control model for irreducible rules:
#    Stack ~H layers of width-3 convs with nonlinearities (or use dilations)
#    to emulate composition. Useful to confirm the depth bottleneck.
# =============================================================================


In [128]:
# plotting
def plot_spacetime(grid: torch.Tensor, title: str, outpath: Path, dpi=220):
    """
    Space–time diagram: rows=time (top=0), columns=space; values in {0,1}.
    """
    outpath.parent.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.imshow(grid.numpy(), aspect='auto', interpolation='nearest', cmap='binary')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Space")
    ax.set_ylabel("Time")
    ax.set_xticks([]); ax.set_yticks([])
    fig.tight_layout()
    fig.savefig(outpath, dpi=dpi)
    plt.close(fig)

def plot_k_mask(mask: np.ndarray, title: str, outpath: Path, dpi=220):
    """
    Visualize a binary mask K^(H) (1D) as a thin horizontal image.
    mask: shape [2H+1], entries in {0,1}.
    """
    outpath.parent.mkdir(parents=True, exist_ok=True)
    img = mask[np.newaxis, :]  # make it 2D: [1, W]
    fig, ax = plt.subplots(figsize=(6, 1.0))
    ax.imshow(img, aspect='auto', interpolation='nearest', cmap='binary')
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    fig.tight_layout()
    fig.savefig(outpath, dpi=dpi, bbox_inches='tight', pad_inches=0.05)
    plt.close(fig)

def kernel_power_gf2(base: np.ndarray, H: int) -> np.ndarray:
    """
    Repeated convolution over GF(2) of a base kernel (e.g., [1,1,1] for Rule 150),
    producing K^(H) of length (2H+1).
    """
    if H == 0:
        return np.array([1], dtype=np.uint8)
    k = base.astype(np.uint8)
    acc = base.astype(np.uint8)
    for _ in range(1, H):
        # ordinary integer convolution, then reduce mod 2
        acc = np.convolve(acc, k) % 2
    return acc.astype(np.uint8)

def k_mask_rule(rule: int, H: int) -> np.ndarray:
    """
    Return K^(H) for Rule 150 or Rule 90 via GF(2) convolution.
    (For other rules this "single-step kernel" shortcut generally doesn't exist.)
    """
    if rule == 150:
        base = np.array([1, 1, 1], dtype=np.uint8)  # z^{-1} + 1 + z
    elif rule == 90:
        base = np.array([1, 0, 1], dtype=np.uint8)  # z^{-1} + z
    else:
        raise ValueError("k_mask_rule only defined for Rules 150 and 90.")
    return kernel_power_gf2(base, H)

In [129]:
# Parameters:
H = 128

In [130]:

# Example runs:
train_rule(150, H=H)   # should climb to ~1.00 quickly


NameError: name 'x_bits' is not defined

In [ ]:
# Example runs:
train_rule(90, H=H)   # should also get ~1.00

rule  90 | H=128 | step  25 | loss 0.017 | acc 1.000
rule  90 | H=128 | step  50 | loss 0.004 | acc 1.000
rule  90 | H=128 | step  75 | loss 0.002 | acc 1.000
rule  90 | H=128 | step 100 | loss 0.002 | acc 1.000
rule  90 | H=128 | step 125 | loss 0.001 | acc 1.000
rule  90 | H=128 | step 150 | loss 0.001 | acc 1.000
rule  90 | H=128 | step 175 | loss 0.001 | acc 1.000
rule  90 | H=128 | step 200 | loss 0.001 | acc 1.000
rule  90 | H=128 | step 225 | loss 0.001 | acc 1.000
rule  90 | H=128 | step 250 | loss 0.001 | acc 1.000
rule  90 | H=128 | step 275 | loss 0.000 | acc 1.000
rule  90 | H=128 | step 300 | loss 0.000 | acc 1.000
rule  90 | H=128 | step 325 | loss 0.000 | acc 1.000
rule  90 | H=128 | step 350 | loss 0.000 | acc 1.000
rule  90 | H=128 | step 375 | loss 0.000 | acc 1.000
rule  90 | H=128 | step 400 | loss 0.000 | acc 1.000
rule  90 | H=128 | step 425 | loss 0.000 | acc 1.000
rule  90 | H=128 | step 450 | loss 0.000 | acc 1.000
rule  90 | H=128 | step 475 | loss 0.000 | acc

In [ ]:
# Example runs:

train_rule( 30, H=H)   # should stall far below 1.00; gets worse as H increases

rule  30 | H=128 | step  25 | loss 0.693 | acc 0.497
rule  30 | H=128 | step  50 | loss 0.693 | acc 0.499
rule  30 | H=128 | step  75 | loss 0.693 | acc 0.496
rule  30 | H=128 | step 100 | loss 0.693 | acc 0.500
rule  30 | H=128 | step 125 | loss 0.693 | acc 0.504
rule  30 | H=128 | step 150 | loss 0.693 | acc 0.503
rule  30 | H=128 | step 175 | loss 0.693 | acc 0.497
rule  30 | H=128 | step 200 | loss 0.693 | acc 0.498
rule  30 | H=128 | step 225 | loss 0.693 | acc 0.503
rule  30 | H=128 | step 250 | loss 0.693 | acc 0.506
rule  30 | H=128 | step 275 | loss 0.693 | acc 0.495
rule  30 | H=128 | step 300 | loss 0.693 | acc 0.503
rule  30 | H=128 | step 325 | loss 0.693 | acc 0.500
rule  30 | H=128 | step 350 | loss 0.693 | acc 0.499
rule  30 | H=128 | step 375 | loss 0.693 | acc 0.499
rule  30 | H=128 | step 400 | loss 0.693 | acc 0.495
rule  30 | H=128 | step 425 | loss 0.693 | acc 0.502
rule  30 | H=128 | step 450 | loss 0.693 | acc 0.507
rule  30 | H=128 | step 475 | loss 0.693 | acc

In [ ]:
# Example runs:

train_rule( 110, H=H)   # should stall far below 1.00; gets worse as H increases

rule 110 | H=128 | step  25 | loss 0.685 | acc 0.569
rule 110 | H=128 | step  50 | loss 0.685 | acc 0.566
rule 110 | H=128 | step  75 | loss 0.683 | acc 0.571
rule 110 | H=128 | step 100 | loss 0.683 | acc 0.573
rule 110 | H=128 | step 125 | loss 0.684 | acc 0.568
rule 110 | H=128 | step 150 | loss 0.683 | acc 0.569
rule 110 | H=128 | step 175 | loss 0.684 | acc 0.567
rule 110 | H=128 | step 200 | loss 0.683 | acc 0.572
rule 110 | H=128 | step 225 | loss 0.684 | acc 0.569
rule 110 | H=128 | step 250 | loss 0.683 | acc 0.570
rule 110 | H=128 | step 275 | loss 0.683 | acc 0.571
rule 110 | H=128 | step 300 | loss 0.684 | acc 0.569
rule 110 | H=128 | step 325 | loss 0.683 | acc 0.570
rule 110 | H=128 | step 350 | loss 0.683 | acc 0.571
rule 110 | H=128 | step 375 | loss 0.684 | acc 0.568
rule 110 | H=128 | step 400 | loss 0.683 | acc 0.571
rule 110 | H=128 | step 425 | loss 0.683 | acc 0.570
rule 110 | H=128 | step 450 | loss 0.685 | acc 0.564
rule 110 | H=128 | step 475 | loss 0.684 | acc

In [ ]:
# --- Generate and save figures ---
out_files = []
settings = dict(width=256, steps=256, p_init=0.5, seed=42, init="random")

for rule, name in [(150, "Rule 150 (reducible)"),
                   ( 90, "Rule 90 (reducible)"),
                   ( 30, "Rule 30 (irreducible)")]:

    grid = simulate(rule, **settings)
    out = plot_spacetime(grid, f"{name}: random IC", f"/mnt/data/rule{rule}_spacetime_random.png")
    out_files.append(out)

# Also make single-seed (single 1 in the center) versions (useful for 150/90 triangles)
single_settings = dict(width=256, steps=256, seed=1, init="single")

for rule, name in [(150, "Rule 150 (reducible)"),
                   ( 90, "Rule 90 (reducible)"),
                   ( 30, "Rule 30 (irreducible)")]:

    grid = simulate(rule, **single_settings)
    out = plot_spacetime(grid, f"{name}: single-cell IC", f"/mnt/data/rule{rule}_spacetime_single.png")
    out_files.append(out)

out_files

AttributeError: 'str' object has no attribute 'parent'